# Local Cleanup

> **Notebook flow:** 01 Setup → 02 EDA → 03 Train → 04 Docker → 05 Kubernetes → **[06 Cleanup]** · 07 Azure Deploy · 08 API Tests

Removes all local resources created during the local development and testing notebooks.

Run sections selectively based on what you have created, or run all cells top-to-bottom for a full teardown.

---

## Resources covered

| Resource | Created by | Section |
|---|---|---|
| `bm-smoke-test` container | `04_docker_testing.ipynb` | §1 |
| `bank-marketing-api:local` Docker image (inference) | `04_docker_testing.ipynb` | §2 |
| `bank-marketing-train:local` Docker image (training) | `04_docker_testing.ipynb` | §2 |
| kind cluster `bm-local` + namespaces + workloads | `05_kubernetes_setup.ipynb` | §3 |
| `artifacts/model.pkl`, `artifacts/metrics.json` | `03_ml_pipeline.ipynb` / `04_docker_testing.ipynb` | §4 |
| `data/results/predictions.csv` | `03_ml_pipeline.ipynb` | §4 |
| `kubeconform` binary | `05_kubernetes_setup.ipynb` | §5 (optional) |
| `kubectl` binary | `05_kubernetes_setup.ipynb` | §5 (optional) |
| `kind` binary | `05_kubernetes_setup.ipynb` | §5 (optional) |


## 1. Stop and Remove Docker Container

Stops the `bm-smoke-test` container if it is still running.

In [ ]:
%%bash
if docker ps -a --format '{{.Names}}' | grep -q '^bm-smoke-test$'; then
  docker stop bm-smoke-test 2>/dev/null || true
  docker rm bm-smoke-test
  echo 'Container bm-smoke-test removed.'
else
  echo 'Container bm-smoke-test not found — nothing to remove.'
fi

## 2. Remove Docker Images

Removes the locally built inference (`bank-marketing-api:local`) and training (`bank-marketing-train:local`) images.

In [ ]:
%%bash
for image in bank-marketing-api:local bank-marketing-train:local; do
  if docker image inspect "$image" &>/dev/null; then
    docker rmi "$image"
    echo "  Removed $image"
  else
    echo "  $image not found — nothing to remove."
  fi
done

## 3. Delete kind Cluster

Deletes the `bm-local` kind cluster and all namespaces/workloads inside it.

> This removes the `bank-marketing` (production) and `bank-marketing-dev` (staging) namespaces and everything deployed to them.

In [ ]:
%%bash
if command -v kind &>/dev/null && kind get clusters 2>/dev/null | grep -q '^bm-local$'; then
  kind delete cluster --name bm-local
  echo 'kind cluster bm-local deleted.'
else
  echo 'kind cluster bm-local not found — nothing to remove.'
fi

## 4. Remove ML Artifacts and Prediction Output

Removes the trained model artifact, metrics file, and batch prediction CSV.

> `.gitkeep` files are preserved so the directory structure stays intact.

In [ ]:
import os

ROOT = "/workspaces/marketing-model-mlops-azure"
targets = [
    "artifacts/model.pkl",
    "artifacts/metrics.json",
    "data/results/predictions.csv",
]

for rel_path in targets:
    full = os.path.join(ROOT, rel_path)
    if os.path.exists(full):
        os.remove(full)
        print(f"  Removed  {rel_path}")
    else:
        print(f"  Not found  {rel_path}")

print()
print("Artifacts cleaned.")

## 5. Remove Installed Binaries (Optional)

Removes `kubectl`, `kind`, and `kubeconform` from `/usr/local/bin/`.

> **Skip this section** if you want to keep these tools available for future runs. They will be re-installed automatically by `05_kubernetes_setup.ipynb` if missing.

In [ ]:
%%bash
for bin in kubectl kind kubeconform; do
  if command -v $bin &>/dev/null; then
    sudo rm /usr/local/bin/$bin
    echo "  Removed $bin"
  else
    echo "  $bin not found — skipping"
  fi
done

## 6. Verify Clean State

Confirm all resources have been removed.

In [ ]:
%%bash
echo '=== Docker container ==='
docker ps -a --filter name=bm-smoke-test --format 'table {{.Names}}\t{{.Status}}' 2>/dev/null \
  || echo '  (docker not available)'
echo ''

echo '=== Docker images ==='
docker images bank-marketing-api:local 2>/dev/null \
  || echo '  (docker not available)'
docker images bank-marketing-train:local 2>/dev/null \
  || echo '  (docker not available)'
echo ''

echo '=== kind clusters ==='
kind get clusters 2>/dev/null || echo '  (kind not installed or no clusters)'
echo ''

echo '=== ML artifacts ==='
ls -lh /workspaces/marketing-model-mlops-azure/artifacts/ 2>/dev/null
ls -lh /workspaces/marketing-model-mlops-azure/data/results/ 2>/dev/null
echo ''

echo '=== Installed binaries ==='
for bin in kubectl kind kubeconform; do
  command -v $bin &>/dev/null && echo "  $bin: $(which $bin)" || echo "  $bin: not installed"
done

---

## Summary

| Resource | Expected state after cleanup |
|---|---|
| `bm-smoke-test` container | Removed |
| `bank-marketing-api:local` image (inference) | Removed |
| `bank-marketing-train:local` image (training) | Removed |
| kind cluster `bm-local` | Deleted |
| `artifacts/model.pkl` | Removed |
| `artifacts/metrics.json` | Removed |
| `data/results/predictions.csv` | Removed |
| `kubectl`, `kind`, `kubeconform` | Removed (if §5 was run) |

To restart local testing from scratch, run the notebooks in order: `01` → `02` → `03` → `04` → `05`.